In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# Core: coupling index
# ----------------------------
def S(pa, r):
    """Dimensionless coupling index: sign matches Cov(X_jk, X_jl) for k!=l."""
    return (pa / (pa + 1.0)) * r - 1.0

# Moment-ratio r = E[λ^2]/E[λ]^2 for common positive priors (scale cancels)
def r_half_normal():
    # λ = |N(0,1)|: E[λ]=sqrt(2/pi), E[λ^2]=1 => r = 1/(2/pi) = pi/2
    return np.pi / 2.0

def r_half_laplace():
    # Half-Laplace = Exponential (any scale): r = 2
    return 2.0

def r_lognormal(sigma):
    # log λ ~ N(0, sigma^2): r = exp(sigma^2)
    return np.exp(sigma**2)

# Monte Carlo estimate for half-t
def r_half_t(nu, n=200_000, seed=0):
    if nu <= 2:
        return np.inf
    rng = np.random.default_rng(seed)
    lam = np.abs(rng.standard_t(df=nu, size=n))
    m1 = lam.mean()
    m2 = (lam**2).mean()
    return m2 / (m1*m1)

# ----------------------------
# Choose what to show (keep it minimal)
# ----------------------------
pa = np.linspace(0.05, 10.0, 500)

curves = [
    ("Half-normal", r_half_normal()),
    ("Half-Laplace", r_half_laplace()),
    #(r"Lognormal $\sigma=0.6$", r_lognormal(0.6)),
    (r"Lognormal $\sigma=1.0$", r_lognormal(1.0)),
    (r"Half-$t_{\nu=3}$", r_half_t(3, n=250_000, seed=3)),
    (r"Half-$t_{\nu=10}$", r_half_t(10, n=250_000, seed=10)),
]

# ----------------------------
# Plot
# ----------------------------
plt.figure(figsize=(9, 5))

# Background shading: negative vs positive covariance
plt.axhline(0, linestyle="--", linewidth=1.8)
plt.fill_between(pa, -2.5, 0, alpha=0.12)  # Cov<0 region
plt.fill_between(pa, 0, 2.5, alpha=0.06)   # Cov>0 region

# Plot curves
for name, r in curves:
    y = S(pa, r)
    plt.plot(pa, y, linewidth=2, label=name)


# Cosmetic / axis
plt.xlim(pa[0], pa[-1] * 1.18)  # extra space for labels
plt.ylim(-1.6, 1.6)
plt.xlabel(r"$p\alpha$")
plt.ylabel(r"$S(pa)$")
plt.title(r"Prior on $\lambda$ controls dependence")

# Small annotations that explain how to read it
plt.text(0.1, 0.92, "Cov>0", transform=plt.gca().transAxes, fontsize=10)
plt.text(0.1, 0.08, "Cov<0", transform=plt.gca().transAxes, fontsize=10)

plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.legend()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# palette = {
#     "threshold":  "#000000",  # black reference
#     "halfnormal": "#E69F00",  # orange
#     "lognormal":  "#009E73",  # bluish green
#     "gamma":      "#D55E00",  # vermillion
#     "invgamma":   "#CC79A7",  # reddish purple
#     "betaprime":  "#0072B2",  # blue
#     "halft":      "#56B4E9",  # sky blue
# }
palette = {
    "threshold":  "#000000",

    "halfnormal": "darkgoldenrod",

    # Separate these two strongly (hue + luminance)
    "lognormal":  "limegreen",  # much darker green
    "gamma":      "lightcoral",  # brighter red than before

    # These three were too similar before:
    "invgamma":   "mediumslateblue",  # brighter purple (more chroma)
    "betaprime":  "sienna",  # mustard / ochre (not orange)
    "halft":      "mediumvioletred",  # deep pine green (reads green, not cyan)
}





# ----------------------------
# Moment ratio r(lambda)
# ----------------------------

def v_half_normal(ones):
    return ones*np.pi / 2.0 - 1

def v_lognormal(sigma):
    # log lambda ~ N(0, sigma^2)
    return np.exp(sigma**2) - 1

def v_gamma_shape(k):
    # Gamma(shape=k, rate arbitrary)
    return 1.0 + 1.0 / k - 1

def v_inv_gamma_shape(a):
    # Inverse-Gamma(shape=a), valid for a>2
    r = np.full_like(a, np.nan, dtype=float)
    mask = a > 2
    r[mask] = (a[mask] - 1.0) / (a[mask] - 2.0) - 1
    return r

def v_beta_prime(a, b):
    # lambda ~ BetaPrime(a,b), b>2
    v = np.full_like(b, np.nan, dtype=float)
    mask = b > 2
    v[mask] = (a + b[mask] - 1.0) / (a * (b[mask] - 2.0))
    return v

def v_half_t_mc(nu, n=250_000, seed=1):
    # Half-Student-t via Monte Carlo (nu>2)
    rng = np.random.default_rng(seed)
    rvals = []
    for df in nu:
        if df <= 1:
            rvals.append(np.inf)
        else:
            lam = np.abs(rng.standard_t(df=df, size=n))
            rvals.append((lam**2).mean() / (lam.mean()**2) - 1) 
    return np.array(rvals)

# ----------------------------
# Parameter grids
# ----------------------------
ones = np.ones(300)
sigma_halfnormal = np.linspace(0.0, 10, 300)       # lognormal
sigma_lognormal = np.linspace(0.0, 1.5, 300)       # lognormal
k     = np.linspace(0.2, 10.0, 300)      # gamma
a     = np.linspace(2.05, 12.0, 300)     # inverse-gamma
#nu    = np.array([2, 2.25, 2.5, 2.75, 3, 3.25, 3.5, 4, 4.5, 5, 5.5, 6, 7, 8, 9, 10])         # half-t (points)
nu    = np.linspace(2.0, 7.0, 20)
a0 = 1.0                         # fixed shape
b_beta = np.linspace(2.05, 7.0, 300)   # tail parameter (must be > 2)
v_ht = v_half_t_mc(nu)

# ----------------------------
# Choose p*alpha for reference
# ----------------------------

pa = 1.0                  # change to what you use in the paper
r_star = 1.0 + 1.0 / pa   # covariance flip threshold
v_star = 1.0 / pa

# ----------------------------
# Plot
# ----------------------------

plt.figure(figsize=(7, 5))

# Curves
# Curves: label by distribution + tail parameter only
plt.plot(sigma_halfnormal, v_half_normal(ones),
         linewidth=2, color=palette['halfnormal'],
         label=r"Half-Normal", solid_capstyle="round")

plt.plot(sigma_lognormal, v_lognormal(sigma_lognormal),
         linewidth=2, color=palette["lognormal"],
         label=r"Lognormal ($\sigma$)", solid_capstyle="round")

plt.plot(k, v_gamma_shape(k),
         linewidth=2, color=palette['gamma'],
         label=r"Gamma (shape $k$)", solid_capstyle="round")

plt.plot(a, v_inv_gamma_shape(a),
         linewidth=2, color=palette['invgamma'],
         label=r"Inverse-Gamma (shape $a$)", solid_capstyle="round")

# plt.scatter(b_beta, v_beta_prime(a0, b_beta),
#          s=60, zorder=3,
#          color=palette["betaprime"],
#          label=r"Beta-prime ($b$, $a=1$)")

plt.plot(b_beta, v_beta_prime(a0, b_beta),
         linewidth=2,
         color=palette["betaprime"],
         label=r"Beta-prime ($b$, $a=1$)", solid_capstyle="round")

plt.scatter(nu, v_ht,
            s=60, zorder=3, color=palette["halft"],
            label=r"Half-$t$ (df $\nu$)")
plt.plot(nu, v_ht,
            linewidth=2, color=palette["halft"],
            linestyle="--", solid_capstyle="round")

# Threshold line: explain in caption or text, not formula here
plt.axhline(v_star, linestyle="--", linewidth=2, color=palette['threshold'])

# Shading: very plain language
plt.fill_between([0, 30], 0.0, v_star, color = "#DCEBFA",
                 alpha=0.35, zorder=0)#, label="Cov<0")
plt.fill_between([0, 30], v_star, 20.0, color = "#F9D6D5",
                 alpha=0.35, zorder=0)#, label="Cov>0")

plt.text(0.22, 0.10, "Cov < 0", transform=plt.gca().transAxes,
         fontsize=14, color="#1f77b4", alpha=0.9)
plt.text(0.22, 0.15, "Cov > 0", transform=plt.gca().transAxes,
         fontsize=14, color="#D55E00", alpha=0.9)

# Cosmetics
plt.xlabel("Tail parameter", fontsize=15)
plt.ylabel(r"$r(\lambda)$", fontsize=20)#=\frac{Var(\lambda)}{\mathbb{E}[\lambda]^2}$", fontsize=15)
plt.title(r"")
plt.ylim(0.0, 7.0)
plt.xlim(0, 6)
plt.grid(True, alpha=0.3)
plt.legend(loc="upper right", frameon=True, fontsize=12)
plt.savefig("figures_for_use_in_paper/covariance_structure.pdf", bbox_inches="tight")
plt.tight_layout()
plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

palette = {
    "threshold":  "#000000",  # black reference
    "halfnormal": "#E69F00",  # orange
    "lognormal":  "#009E73",  # bluish green
    "gamma":      "#D55E00",  # vermillion
    "invgamma":   "#CC79A7",  # reddish purple
    "betaprime":  "#0072B2",  # blue
    "halft":      "#56B4E9",  # sky blue
}

# ----------------------------
# Moment ratio r(lambda^2)
# ----------------------------

def v2_half_normal(ones):
    # lambda ~ HalfNormal(1)
    # E[λ^2]=1, Var(λ^2)=2
    return 2.0 * ones

def v2_lognormal(sigma):
    # log λ ~ N(0, σ^2)
    # E[λ^2]=exp(2σ^2), Var(λ^2)=exp(4σ^2)(exp(4σ^2)-1)
    return np.exp(4 * sigma**2) - 1

def v2_gamma_shape(k):
    # λ ~ Gamma(shape=k, rate arbitrary)
    # Var(λ^2)/E[λ^2]^2 = 2(2k+3)/(k(k+1))
    return 2.0 * (2.0 * k + 3.0) / (k * (k + 1.0))

def v2_inv_gamma_shape(a):
    # λ ~ Inv-Gamma(shape=a), requires a>4
    r = np.full_like(a, np.nan, dtype=float)
    mask = a > 4
    r[mask] = (
        2.0 * (2.0 * a[mask] - 3.0)
        / ((a[mask] - 4.0) * (a[mask] - 3.0))
    )
    return r

def v2_beta_prime(a, b):
    # λ ~ BetaPrime(a,b), requires b>4
    r = np.full_like(b, np.nan, dtype=float)
    mask = b > 4
    r[mask] = (
        2.0 * (a + b[mask] - 1.0) * (2.0 * a + b[mask] - 2.0)
        / ((b[mask] - 4.0) * (b[mask] - 3.0) * a * (a + 1.0))
    )
    return r

def v2_half_t_mc(nu, n=250_000, seed=1):
    # Monte Carlo for λ ~ |t_ν|, needs ν>4
    rng = np.random.default_rng(seed)
    rvals = []
    for df in nu:
        if df <= 4:
            rvals.append(np.inf)
        else:
            lam = np.abs(rng.standard_t(df=df, size=n))
            rvals.append(
                (lam**4).mean() / ((lam**2).mean()**2) - 1
            )
    return np.array(rvals)

# ----------------------------
# Parameter grids
# ----------------------------
ones = np.ones(300)
sigma_halfnormal = np.linspace(0.0, 20, 300)
sigma_lognormal = np.linspace(0.0, 1.5, 300)
k = np.linspace(0.3, 20.0, 300)
a = np.linspace(4.05, 20.0, 300)
nu = np.linspace(4.1, 20.0, 40)

a0 = 1.0
b_beta = np.linspace(4.05, 20.0, 300)

v_ht = v2_half_t_mc(nu)

# ----------------------------
# Choose p*alpha for reference
# ----------------------------
p=10
alpha=0.1
pa = p*alpha
print(pa)
v_star = 1.0 / pa   # same threshold, now for λ^2

# ----------------------------
# Plot
# ----------------------------
plt.figure(figsize=(8.5, 5))

plt.plot(sigma_halfnormal, v2_half_normal(ones),
         linewidth=2, color=palette['halfnormal'],
         label=r"Half-Normal")

plt.plot(sigma_lognormal, v2_lognormal(sigma_lognormal),
         linewidth=2, color=palette["lognormal"],
         label=r"Lognormal ($\sigma$)")

plt.plot(k, v2_gamma_shape(k),
         linewidth=2, color=palette['gamma'],
         label=r"Gamma (shape $\alpha$)")

plt.plot(a, v2_inv_gamma_shape(a),
         linewidth=2, color=palette['invgamma'],
         label=r"Inverse-Gamma (shape $\alpha$)")

plt.plot(b_beta, v2_beta_prime(a0, b_beta),
         linewidth=2, linestyle=":",
         color=palette["betaprime"],
         label=r"Beta-prime ($b$, $a=1$)")

plt.scatter(nu, v_ht,
            s=60, zorder=3, color=palette["halft"],
            label=r"Half-$t$ (df $\nu$)")
plt.plot(nu, v_ht,
         linewidth=2, color=palette["halft"],
         linestyle="--")

plt.axhline(v_star, linestyle="--", linewidth=2,
            color=palette['threshold'], label=r"$p\alpha$")

plt.fill_between([0, 30], 0.0, v_star,
                 color="#DCEBFA", alpha=0.35, zorder=0)
plt.fill_between([0, 30], v_star, 20.0,
                 color="#F9D6D5", alpha=0.35, zorder=0)

plt.text(0.22, 0.10, "Cov < 0", transform=plt.gca().transAxes,
         fontsize=11, color="#1f77b4", alpha=0.9)
plt.text(0.22, 0.15, "Cov > 0", transform=plt.gca().transAxes,
         fontsize=11, color="#D55E00", alpha=0.9)

plt.xlabel("tail parameter")
plt.ylabel(r"$r(\lambda^2)=\frac{Var(\lambda^2)}{\mathbb{E}[\lambda^2]^2}$")
plt.ylim(0.0, 7.0)
plt.xlim(0, 7.2)
plt.grid(True, alpha=0.3)
plt.legend(loc="upper right", frameon=True)

plt.savefig("figures_for_use_in_paper/covariance_structure_lambda_squared.pdf",
            bbox_inches="tight")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

palette = {
    "threshold":  "#000000",  # black reference
    "halfnormal": "#E69F00",  # orange
    "lognormal":  "#009E73",  # bluish green
    "gamma":      "#D55E00",  # vermillion
    "invgamma":   "#CC79A7",  # reddish purple
    "betaprime":  "#0072B2",  # blue
    "halft":      "#56B4E9",  # sky blue
}

# ----------------------------
# Regularization map
# ----------------------------
def lambda_tilde_sq(lam, tau, c):
    # \tilde{\lambda}^2 = c^2 * lam^2 / (c^2 + tau^2 * lam^2)
    lam2 = lam**2
    return (c**2 * lam2) / (c**2 + (tau**2) * lam2)

def r_from_samples(x):
    # r(x) = Var(x) / E[x]^2
    m = np.mean(x)
    v = np.var(x)
    return v / (m**2)

# ----------------------------
# Samplers for lambda
# (Rates/scales chosen for simplicity; the ratio is scale-invariant BEFORE regularization,
#  but AFTER regularization the absolute scale matters through (tau,c).)
# ----------------------------
def sample_half_normal(rng, n):
    return np.abs(rng.normal(loc=0.0, scale=1.0, size=n))

def sample_lognormal(rng, n, sigma):
    # log lam ~ N(0, sigma^2)
    return np.exp(rng.normal(loc=0.0, scale=sigma, size=n))

def sample_gamma_shape(rng, n, k):
    # Gamma(shape=k, rate=1)
    return rng.gamma(shape=k, scale=1.0, size=n)

def sample_inv_gamma_shape(rng, n, a):
    # Inv-Gamma(shape=a, scale=1) implemented as 1 / Gamma(a, rate=1)
    g = rng.gamma(shape=a, scale=1.0, size=n)
    return 1.0 / g

def sample_beta_prime(rng, n, a, b):
    # BetaPrime(a,b): if U~Beta(a,b), then U/(1-U)
    u = rng.beta(a=a, b=b, size=n)
    return u / (1.0 - u)

def sample_half_t(rng, n, nu):
    return np.abs(rng.standard_t(df=nu, size=n))

# ----------------------------
# MC curves for r(\tilde{lambda}^2)
# ----------------------------
def mc_curve(param_grid, sampler_fn, tau, c, n=200_000, seed=1):
    rng = np.random.default_rng(seed)
    out = np.empty_like(param_grid, dtype=float)

    for i, p in enumerate(param_grid):
        lam = sampler_fn(rng, n, p)
        lt2 = lambda_tilde_sq(lam, tau=tau, c=c)
        out[i] = r_from_samples(lt2)

    return out


# Helper wrappers so mc_curve can always pass the varying parameter p
def sampler_half_normal(rng, n, _):
    return np.abs(rng.normal(loc=0.0, scale=1.0, size=n))

def sampler_lognormal(rng, n, sigma):
    return np.exp(rng.normal(loc=0.0, scale=sigma, size=n))

def sampler_gamma(rng, n, k):
    return rng.gamma(shape=k, scale=1.0, size=n)

def sampler_invgamma(rng, n, a):
    g = rng.gamma(shape=a, scale=1.0, size=n)
    return 1.0 / g

def sampler_betaprime(rng, n, b, a_fixed=1.0):
    u = rng.beta(a=a_fixed, b=b, size=n)
    return u / (1.0 - u)

def sampler_halft(rng, n, nu):
    return np.abs(rng.standard_t(df=nu, size=n))

# ----------------------------
# Parameter grids
# ----------------------------
ones = np.ones(300)

sigma_halfnormal = np.linspace(0.0, 10, 300)   # just used as x-axis placeholder for HalfNormal line
sigma_lognormal  = np.linspace(0.0, 10, 300)
k_grid           = np.linspace(0.2, 10.0, 300)
a_grid           = np.linspace(0.2, 12.0, 300)
b_beta_grid      = np.linspace(0.2, 12.0, 300)
nu_grid          = np.linspace(0.2, 10.0, 30)

# ----------------------------
# Choose tau, c for regularization
# ----------------------------
tau = 1.0   # keep fixed as you said
c   = np.sqrt(4.0)   # choose your regularization cap (prior SD cap is ~c)

# ----------------------------
# Compute curves (MC)
# ----------------------------
# Half-normal has no tail parameter; we compute a flat line by MC once
rng0 = np.random.default_rng(123)
lam_hn = sample_half_normal(rng0, 300_000)
v_hn = r_from_samples(lambda_tilde_sq(lam_hn, tau=tau, c=c)) * ones

v_hn  = mc_curve(sigma_halfnormal, sampler_half_normal, tau=tau, c=c, seed=1)
v_ln  = mc_curve(sigma_lognormal,  sampler_lognormal,  tau=tau, c=c, seed=2)
v_gam = mc_curve(k_grid,           sampler_gamma,      tau=tau, c=c, seed=3)
v_ig  = mc_curve(a_grid,           sampler_invgamma,   tau=tau, c=c, seed=4)

v_bp = np.empty_like(b_beta_grid)
rng_bp = np.random.default_rng(5)

for i, b in enumerate(b_beta_grid):
    lam = sampler_betaprime(rng_bp, 200_000, b)
    v_bp[i] = r_from_samples(lambda_tilde_sq(lam, tau=tau, c=c))

v_ht = mc_curve(nu_grid, sampler_halft, tau=tau, c=c, n=250_000, seed=6)

In [ ]:

# ----------------------------
# Threshold 1/(p alpha)
# ----------------------------
p=10
alpha=0.1
pa = p*alpha
v_star = 1.0 / pa

# ----------------------------
# Plot
# ----------------------------
plt.figure(figsize=(8.5, 5))

plt.plot(sigma_halfnormal, v_hn,
         linewidth=2, color=palette['halfnormal'],
         label=r"Half-Normal")

plt.plot(sigma_lognormal, v_ln,
         linewidth=2, color=palette["lognormal"],
         label=r"Lognormal ($\sigma$)")

plt.plot(k_grid, v_gam,
         linewidth=2, color=palette['gamma'],
         label=r"Gamma (shape $\alpha$)")

plt.plot(a_grid, v_ig,
         linewidth=2, color=palette['invgamma'],
         label=r"Inverse-Gamma (shape $\alpha$)")

plt.plot(b_beta_grid, v_bp,
         linewidth=2, linestyle=":",
         color=palette["betaprime"],
         label=r"Beta-prime ($b$, $a=1$)")

plt.scatter(nu_grid, v_ht,
            s=60, zorder=3, color=palette["halft"],
            label=r"Half-$t$ (df $\nu$)")
plt.plot(nu_grid, v_ht,
         linewidth=2, color=palette["halft"],
         linestyle="--")

plt.axhline(v_star, linestyle="--", linewidth=2,
            color=palette['threshold'], label=r"$p\alpha$")

plt.fill_between([0, 30], 0.0, v_star,
                 color="#DCEBFA", alpha=0.35, zorder=0)
plt.fill_between([0, 30], v_star, 20.0,
                 color="#F9D6D5", alpha=0.35, zorder=0)

plt.text(0.22, 0.10, "Cov < 0", transform=plt.gca().transAxes,
         fontsize=11, color="#1f77b4", alpha=0.9)
plt.text(0.22, 0.15, "Cov > 0", transform=plt.gca().transAxes,
         fontsize=11, color="#D55E00", alpha=0.9)

plt.xlabel("tail parameter")
plt.ylabel(r"$r(\tilde{\lambda}^2)=\frac{Var(\tilde{\lambda}^2)}{\mathbb{E}[\tilde{\lambda}^2]^2}$")
plt.ylim(0.0, 5.0)
plt.xlim(0, 5.2)
plt.grid(True, alpha=0.3)
plt.legend(loc="upper right", frameon=True)

plt.savefig("figures_for_use_in_paper/covariance_structure_lambda_squared_regularized.pdf",
            bbox_inches="tight")
plt.tight_layout()
plt.show()


In [6]:
import numpy as np
import matplotlib.pyplot as plt

# If SciPy is available, we’ll use exact quantiles for Inv-Gamma.
# Otherwise we fall back to Monte Carlo (still fine for a plot).
try:
    from scipy.stats import invgamma
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False


palette = {
    "threshold":  "#000000",

    "halfnormal": "darkgoldenrod",

    # Separate these two strongly (hue + luminance)
    "lognormal":  "limegreen",  # much darker green
    "gamma":      "lightcoral",  # brighter red than before

    # These three were too similar before:
    "invgamma":   "mediumslateblue",  # brighter purple (more chroma)
    "betaprime":  "sienna",  # mustard / ochre (not orange)
    "halft":      "mediumvioletred",  # deep pine green (reads green, not cyan)
}

# ----------------------------
# Regularization map
# ----------------------------
def lambda_tilde_sq(lam, tau, c_sq):
    lam2 = lam**2
    return (c_sq * lam2) / (c_sq + (tau**2) * lam2)

def r_from_samples(x):
    m = np.mean(x)
    v = np.var(x)
    return v / (m**2)

# ----------------------------
# Samplers (all accept (rng, n, param))
# ----------------------------
def sampler_half_normal(rng, n, _):
    return np.abs(rng.normal(loc=0.0, scale=1.0, size=n))

def sampler_lognormal(rng, n, sigma):
    return np.exp(rng.normal(loc=0.0, scale=sigma, size=n))

def sampler_gamma(rng, n, k):
    return rng.gamma(shape=k, scale=1.0, size=n)

def sampler_invgamma(rng, n, a):
    g = rng.gamma(shape=a, scale=1.0, size=n)
    return 1.0 / g

def sampler_betaprime(rng, n, b, a_fixed=1.0):
    u = rng.beta(a=a_fixed, b=b, size=n)
    return u / (1.0 - u)

def sampler_halft(rng, n, nu):
    return np.abs(rng.standard_t(df=nu, size=n))

# ----------------------------
# MC curve helper
# ----------------------------
def mc_curve(param_grid, sampler_fn, tau, c_sq, n=200_000, seed=1):
    rng = np.random.default_rng(seed)
    out = np.empty_like(param_grid, dtype=float)
    for i, p in enumerate(param_grid):
        lam = sampler_fn(rng, n, p)
        lt2 = lambda_tilde_sq(lam, tau=tau, c_sq=c_sq)
        out[i] = r_from_samples(lt2)
    return out

# ----------------------------
# Parameter grids
# ----------------------------
sigma_halfnormal = np.linspace(0.0, 10, 300)   # placeholder x-axis for HalfNormal line
sigma_lognormal  = np.linspace(0.0, 10, 300)
k_grid           = np.linspace(0.2, 10.0, 300)
a_grid           = np.linspace(0.2, 12.0, 300)
b_beta_grid      = np.linspace(0.2, 12.0, 300)
nu_grid          = np.linspace(0.2, 10.0, 300)

# ----------------------------
# Choose tau fixed
# ----------------------------
tau = 1.0

# ----------------------------
# c quantiles: c^2 ~ Inv-Gamma(2, 4)
# Using SciPy if possible; otherwise Monte Carlo.
# ----------------------------
q_levels = [0.5, 0.9]
shape_a = 2.0
scale_b = 4.0  # In SciPy invgamma: "scale" corresponds to β; mean = β/(a-1) = 4

if _HAVE_SCIPY:
    c2_q = invgamma.ppf(q_levels, a=shape_a, scale=scale_b)
    c2_values = c2_q
    c2_values = np.append(c2_values, 1e3)
#test
    # c2_values.append()
else:
    rng_c = np.random.default_rng(2024)
    # If c^2 ~ Inv-Gamma(a, scale=b), then c^2 = 1 / Gamma(a, scale=1/b)?? (depends on convention)
    # We'll sample using: if Y ~ Gamma(a, scale=1), then X = b / Y ~ Inv-Gamma(a, scale=b).
    Y = rng_c.gamma(shape=shape_a, scale=1.0, size=2_000_000)
    c2_samps = scale_b / Y
    c2_values = np.quantile(c2_samps, q_levels)

# ----------------------------
# Threshold 1/(p alpha)
# ----------------------------
p = 10
alpha = 0.1
pa = p * alpha
v_star = 1.0 / pa


In [18]:
def precompute_curves(*, c_values, q_levels, tau,
                      sigma_halfnormal, sigma_lognormal, k_grid, a_grid, b_beta_grid, nu_grid):
    """
    Returns a list of dicts, one per panel, with all y-curves precomputed.
    """
    panels = []

    for c, q in zip(c_values, q_levels):
        # Curves
        v_hn  = mc_curve(sigma_halfnormal, sampler_half_normal, tau=tau, c_sq=c, seed=1)
        v_ln  = mc_curve(sigma_lognormal,  sampler_lognormal,  tau=tau, c_sq=c, seed=2)
        v_gam = mc_curve(k_grid,           sampler_gamma,      tau=tau, c_sq=c, seed=3)
        v_ig  = mc_curve(a_grid,           sampler_invgamma,   tau=tau, c_sq=c, seed=4, n=20_000_000)

        # Beta-prime: vary b, keep a fixed (=1)
        v_bp = np.empty_like(b_beta_grid)
        rng_bp = np.random.default_rng(5)
        for i, b in enumerate(b_beta_grid):
            lam = sampler_betaprime(rng_bp, 2_000_000, b)
            v_bp[i] = r_from_samples(lambda_tilde_sq(lam, tau=tau, c_sq=c))

        # Half-t points
        v_ht = mc_curve(nu_grid, sampler_halft, tau=tau, c_sq=c, n=10_000_000, seed=6)

        panels.append({
            "c": c,
            "q": q,
            "v_hn": v_hn,
            "v_ln": v_ln,
            "v_gam": v_gam,
            "v_ig": v_ig,
            "v_bp": v_bp,
            "v_ht": v_ht,
        })

    return panels


In [ ]:
q_levels = np.append(q_levels, "inf")
q_levels

panels = precompute_curves(
    c_values=c2_values, q_levels=q_levels, tau=tau,
    sigma_halfnormal=sigma_halfnormal,
    sigma_lognormal=sigma_lognormal,
    k_grid=k_grid, a_grid=a_grid,
    b_beta_grid=b_beta_grid,
    nu_grid=nu_grid
)


In [32]:
def plot_curves(panels, *, v_star, palette,
               sigma_halfnormal, sigma_lognormal, k_grid, a_grid, b_beta_grid, nu_grid):
    fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True)

    for ax, panel in zip(axes, panels):
        c, q = panel["c"], panel["q"]

        ax.plot(sigma_halfnormal, panel["v_hn"], linewidth=2, color=palette['halfnormal'], label=r"Half-Normal")
        #ax.plot(sigma_lognormal,  panel["v_ln"], linewidth=2, color=palette["lognormal"],  label=r"Lognormal ($\sigma$)")
        ax.plot(k_grid,           panel["v_gam"], linewidth=2, color=palette["lognormal"],     label=r"Gamma (shape $k$)")
        ax.plot(a_grid,           panel["v_ig"],  linewidth=2, color=palette['invgamma'],  label=r"Inv-Gamma (shape $a$)")
        ax.plot(b_beta_grid,      panel["v_bp"],  linewidth=2, color=palette['gamma'], label=r"Beta-prime ($b$, $a=1$)")
        ax.plot(nu_grid,          panel["v_ht"],  linewidth=2, color=palette["halft"], label=r"Half-$t$ (df $\nu$)")

        ax.axhline(v_star, linestyle="--", linewidth=2, color=palette['threshold'])

        ax.fill_between([0, 30], 0.0, v_star, color="#DCEBFA", alpha=0.35, zorder=0)
        ax.fill_between([0, 30], v_star, 20.0, color="#F9D6D5", alpha=0.35, zorder=0)

        ax.set_xlim(0, 10.2)
        ax.set_ylim(0.0, 7.0)
        ax.grid(True, alpha=0.3)
        ax.set_xlabel("Tail parameter", fontsize=20)
        #ax.set_title(rf"$c^2$ {int(100*q)}% quantile -  $c^2 \approx {c:.3f}$")
        #ax.set_title(fr"$c^2$ at quantile {q:.0%} ($c^2 \approx {c:.3f}$)", fontsize=20)
        if float(q) == 0.1:
            ax.set_title(fr"$c^2 = {c:.3f}$", fontsize=20)
        elif float(q) == 0.5:
            ax.set_title(fr"$c^2 = {c:.3f}$", fontsize=20)
        elif float(q) == 0.9:
            ax.set_title(fr"$c^2 = {c:.3f}$", fontsize=20)
        else:
            ax.set_title(fr"$c^2 = 1e3$", fontsize=20)



    axes[0].set_ylabel(r"$CV(\tilde{\lambda}_j^2)^2$", fontsize=25) #=\frac{Var(\tilde{\lambda}^2)}{\mathbb{E}[\tilde{\lambda}^2]^2}$", fontsize=15)

    handles, labels = axes[-1].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right", bbox_to_anchor=[0.68, 0.935], frameon=True, fontsize=14)

    plt.tight_layout()
    return fig, axes


In [ ]:
fig, axes = plot_curves(
    panels,
    v_star=v_star, palette=palette,
    sigma_halfnormal=sigma_halfnormal,
    sigma_lognormal=sigma_lognormal,
    k_grid=k_grid, a_grid=a_grid,
    b_beta_grid=b_beta_grid,
    nu_grid=nu_grid
)

plt.savefig("figures_for_use_in_paper/covariance_structure_lambda_squared_regularized_c_quantiles.pdf",
            bbox_inches="tight")
plt.show()
